[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kasparvonbeelen/contracts/blob/main/2-4-classify-apply_models_seq.ipynb)


# Apply Classifier

In [ ]:
!pip install -q -U datasets transformers evaluate

In [1]:
import torch
from torch.nn.functional import softmax
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [ ]:
!unzip metadata.tsv.zip

In [2]:
processed_data_dir = Path('processed_data/tous')
file_name = processed_data_dir / 'metadata.tsv' # _annotated_distilbert
metadata = pd.read_csv(file_name,sep='\t')
metadata.fillna('', inplace=True)
print(len(metadata))

124495


In [3]:
metadata.columns

Index(['platform', 'year', 'sentence', 'sentence_processed'], dtype='object')

In [4]:
clause_type = 'modification' # 'opt-out' | 'arbitration' | 'class waiver' | 'anti-scraping' | 'modification'

# produced by 2-3-classify-train_models_seq.ipynb (ModernBERT-base, trained on
# target sentence + context, not a bare-sentence classifier)
model_dir = f'Kaspar/{clause_type}_best_model_modernbert'
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
tokenizer = AutoTokenizer.from_pretrained(model_dir)  # already knows the [TGT] special token added during training

device = (
    "cuda" if torch.cuda.is_available()          # Colab GPU
    else "mps" if torch.backends.mps.is_available()  # macbook
    else "cpu"
)
model.to(device)
model.eval()
print(device)

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/3.58M [00:00<?, ?B/s]

mps


In [5]:
metadata.head()

,platform,year,sentence,sentence_processed
0,twitter,20150501,Twitter Terms of Service These Terms of Servic...,[mask] [mask] [mask] [mask] [mask] [mask] [mas...
1,twitter,20150501,Your access to and use of the Services are con...,your access to and use of the services are con...
2,twitter,20150501,By accessing or using the Services you agree t...,by accessing or using the services you agree t...
3,twitter,20150501,Basic Terms You are responsible for your use o...,basic terms you are responsible for your use o...
4,twitter,20150501,"Most Content you submit, post, or display thro...","most content you submit , post , or display th..."


In [6]:
# The classifier was trained on the target sentence *plus its surrounding
# context* (see 2-3-classify-train_models_seq.ipynb), so -- unlike a bare
# sentence classifier -- the same sentence text can get a different
# prediction depending on where it sits in a document. So we can no longer
# dedupe on the raw sentence text; instead build a [TGT] ... [TGT]-wrapped
# context string per row, one document (tou_id) at a time.
metadata['tou_id'] = metadata['platform'].astype(str) + '_' + metadata['year'].astype(str)

sentences = metadata['sentence'].fillna('').tolist()
tou_ids = metadata['tou_id'].tolist()
n = len(metadata)

def build_context_text(i, window=5):
    """Mirrors get_surrounding_sentences (2-1) + build_text (2-3): wraps the
    target sentence in [TGT] ... [TGT] inside up to `window` sentences of
    context on each side, never crossing into a different tou_id."""
    tid = tou_ids[i]
    prev = [sentences[j] for j in range(max(0, i - window), i) if tou_ids[j] == tid]
    nxt = [sentences[j] for j in range(i + 1, min(n, i + 1 + window)) if tou_ids[j] == tid]
    return f"{' '.join(prev)} [TGT] {sentences[i]} [TGT] {' '.join(nxt)}".strip()

metadata['text'] = [build_context_text(i) for i in tqdm(range(n))]

  0%|          | 0/124495 [00:00<?, ?it/s]

In [ ]:
batch_size = 64 if device == "cuda" else 16  # bigger batches on a Colab GPU; smaller on mps/cpu

probs = []
with torch.no_grad():
    for i in tqdm(range(0, n, batch_size)):
        batch_texts = metadata['text'].iloc[i:i + batch_size].tolist()
        inputs = tokenizer(batch_texts, return_tensors='pt', truncation=True, padding=True).to(device)
        logits = model(**inputs).logits
        probs.extend(softmax(logits, dim=1)[:, 1].cpu().tolist())

metadata[f'prob_1_{clause_type}'] = probs

  0%|          | 0/7781 [00:00<?, ?it/s]

In [ ]:
metadata[clause_type] = (metadata[f'prob_1_{clause_type}'] > .5).astype(int)

In [ ]:
output_file_name = processed_data_dir / 'metadata_annotated_modernbert.tsv'
output_file_name

In [ ]:
metadata.to_csv(output_file_name, sep='\t', index=False)

In [ ]:
metadata[clause_type].value_counts()

In [ ]:
metadata.columns

In [ ]:
metadata['year_int'] = metadata.year.apply(lambda x: int(str(x)[:4]))

In [ ]:
import seaborn as sns

data = metadata.groupby(['platform','year_int'])[clause_type].sum().astype(bool).astype(int).unstack().fillna(-1)



#.loc['bumble'].plot(kind='bar')

In [ ]:
import pandas as _pd

def replace_minus_ones_with_prev(X, axis=1, inplace=False):
    """
    Replace -1 entries in a matrix/array with the nearest preceding 0 or 1 along the given axis.
    If there is no preceding non -1 value, the -1 is left unchanged.

    Parameters:
    - X: array-like (numpy array, list of lists, or pandas DataFrame)
    - axis: 1 to replace along rows (left-to-right), 0 to replace along columns (top-to-bottom)
    - inplace: if True and X is a numpy array or DataFrame, modify it in place; otherwise return a new array

    Returns:
    - numpy.ndarray or pandas.DataFrame with replacements applied (unless inplace=True modifies input)
    """


    is_df = _pd is not None and isinstance(X, _pd.DataFrame)
    if is_df:
        arr = X.values
    else:
        arr = X if isinstance(X, (np.ndarray,)) else np.array(X)

    if not inplace:
        arr = arr.copy()

    if axis not in (0, 1):
        raise ValueError("axis must be 0 or 1")

    # iterate over the chosen axis and carry forward the last seen non -1 value
    if axis == 1:
        # rows
        for r in range(arr.shape[0]):
            last = None
            for c in range(arr.shape[1]):
                val = arr[r, c]
                if val != -1:
                    last = val
                elif last is not None:
                    arr[r, c] = last
    else:
        # columns
        for c in range(arr.shape[1]):
            last = None
            for r in range(arr.shape[0]):
                val = arr[r, c]
                if val != -1:
                    last = val
                elif last is not None:
                    arr[r, c] = last

    if is_df:
        if inplace:
            X.iloc[:, :] = arr
            return X
        else:
            return _pd.DataFrame(arr, index=X.index, columns=X.columns)
    else:
        return arr

sns.set(rc={'figure.figsize':(6.7,10.27)})
sns.heatmap(replace_minus_ones_with_prev(data),cbar=False)

In [ ]:
metadata.columns

In [ ]:
#metadata.drop(columns=['logits_arbitration', 'logits_anti-scraping'], inplace=True)

In [ ]:
metadata.columns

In [ ]:
#metadata.sort_values('prob_1', ascending=False).head(10)

In [ ]:
# load the judge-annotated training data (built in 2-3-classify-train_models_seq.ipynb)
# so we can flag, below, which corpus sentences were part of the training set
df_annotations = pd.read_csv(f'annotations/claude_annotations/{clause_type}_annotations.csv')

In [ ]:
df_deduplicated = metadata.drop_duplicates(subset=['sentence'])
df_deduplicated['annotated'] = df_deduplicated.sentence.isin(df_annotations.Target_sentence)
int_labels = [((0.95,1.0),'confident_positive'),( (0.80,.95), 'sure_positive'),((0.60,.80), 'leaning_positive'),
                   ((0.50,.60), 'borderline_positive'),((0.40,.50), 'borderline_negative'),
                   ((0.20,.40), 'leaning_negative'),((0.05,.20), 'sure_negative'),((0.0,.05), 'confident_negative')]
for interval, label in int_labels:


    df_deduplicated.loc[df_deduplicated[f'prob_1_{clause_type}'].between(*interval),'category']  = label


In [ ]:
pd.concat([df_deduplicated[df_deduplicated.category == label].sample(10)
    for _ , label in int_labels], axis=0)[['sentence','category']].to_csv(f'annotations/inference/{clause_type}_automatic_annotations_by_category.csv')


In [ ]:
metadata[metadata[f'prob_1_{clause_type}'] > .5].to_csv(f'annotations/inference/{clause_type}_inference.csv')

## Fin